# 04. Donor and Village Segmentation

Seeded K-Means groups donors by recency/frequency/quantity and villages by category-level demand and fulfillment history. Segments are campaign-planning aids, not labels about people or communities. Current groups come from synthetic data only.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display
from ml.analysis.segmentation import segment_dataset

cwd = Path.cwd()
WORKSPACE = next((path for path in (cwd, *cwd.parents) if (path / "ml" / "analysis" / "segmentation.py").exists()), cwd)
segments = segment_dataset(WORKSPACE / "data" / "synthetic", expected_origin="synthetic", seed=42)
donors = segments["donors"]
villages = segments["villages"]
print("Synthetic donor segments:")
display(donors.groupby("segment").agg(donors=("donor_id", "count"), median_items=("item_count", "median"), median_quantity=("quantity_total", "median")))
print("Synthetic village clusters:")
display(villages[["village_id", "cluster", "dominant_need_category", "total_need_units", "fulfillment_rate"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for segment, group in donors.groupby("segment"):
    axes[0].scatter(group["recency_days"], group["item_count"], label=segment, alpha=0.75)
axes[0].set_title("Donors: recency and item frequency")
axes[0].set_xlabel("Days since last synthetic listing")
axes[0].set_ylabel("Synthetic item count")
axes[0].legend()
axes[1].scatter(villages["total_need_units"], villages["fulfillment_rate"], c=villages["cluster"], cmap="viridis")
axes[1].set_title("Villages: need volume and fulfillment")
axes[1].set_xlabel("Synthetic need units")
axes[1].set_ylabel("Synthetic fulfillment rate")
plt.tight_layout()
plt.show()
print("Conclusion: these seeded groups are computed from synthetic histories only; do not use them to target real people or villages until validated on the pilot.")
print("Assumptions/limitations: K-Means assumes scaled numeric features and roughly compact clusters. Cluster names are relative summaries, not stable identities or causal explanations.")